In [ ]:
!pip install transformers datasets librosa torchaudio accelerate matplotlib captum

###Φόρτωση συνόλου και προετοιμασία


In [ ]:
from datasets import load_dataset, Audio, DatasetDict
from sklearn.model_selection import GroupShuffleSplit
import numpy as np

dataset = load_dataset("AbstractTTS/IEMOCAP", split="train")
dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))

def merge_and_filter(batch):
    emo = batch["major_emotion"]
    if emo == "excited":
        emo = "happy"
    batch["emotion_4class"] = emo
    return batch

dataset = dataset.map(merge_and_filter)
dataset = dataset.filter(lambda b: b["emotion_4class"] in ["angry", "happy", "neutral", "sad"])

def add_speaker(batch):
    parts = batch["file"].split("_")
    session = parts[0][:5]
    speaker_gender = parts[-1][0]
    batch["speaker_id"] = f"{session}_{speaker_gender}"
    return batch

dataset = dataset.map(add_speaker)

labels = sorted(set(dataset["emotion_4class"]))
num_labels = len(labels)
label2id = {name: i for i, name in enumerate(labels)}
id2label = {i: name for i, name in enumerate(labels)}
print("Κλάσεις:", labels)

splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(X=range(len(dataset)), groups=dataset["speaker_id"]))

dataset = DatasetDict({
    "train": dataset.select(train_idx),
    "test": dataset.select(test_idx),
})

train_speakers = set(dataset["train"]["speaker_id"])
test_speakers = set(dataset["test"]["speaker_id"])

###Φιλτράρισμα σχεδόν-σιωπηλών δειγμάτων

In [ ]:
MAX_SAMPLES = 6 * 16_000  # 6 δευτερόλεπτα

def is_valid_audio(batch):
    audio = batch["audio"]["array"][:MAX_SAMPLES]
    if len(audio) == 0:
        return False
    return np.std(audio) >= 1e-4

print("Πριν:", len(dataset["train"]), len(dataset["test"]))
dataset["train"] = dataset["train"].filter(is_valid_audio, load_from_cache_file=False)
dataset["test"] = dataset["test"].filter(is_valid_audio, load_from_cache_file=False)
print("Μετά:", len(dataset["train"]), len(dataset["test"]))

###Φόρτωση μοντέλου

In [ ]:
import torch
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

checkpoint = "microsoft/unispeech-sat-base"

feature_extractor = AutoFeatureExtractor.from_pretrained(checkpoint)
model = AutoModelForAudioClassification.from_pretrained(
    checkpoint, num_labels=num_labels, label2id=label2id, id2label=id2label,
    ignore_mismatched_sizes=True,
).to("cuda")
model.freeze_feature_encoder()

###Προετοιμασία dataset

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]
    audio_array = audio["array"][:MAX_SAMPLES]
    batch["input_values"] = feature_extractor(audio_array, sampling_rate=audio["sampling_rate"]).input_values[0]
    batch["labels"] = label2id[batch["emotion_4class"]]
    return batch

encoded_dataset = dataset.map(prepare_dataset, remove_columns=dataset["train"].column_names, load_from_cache_file=False)

###Ορισμός μετρικών εκπαίδευσης

In [ ]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, cohen_kappa_score, roc_auc_score,
    average_precision_score, log_loss, confusion_matrix,
)
from sklearn.preprocessing import label_binarize
from scipy.special import softmax

all_confusion_matrices = []

def compute_metrics(eval_pred):
    logits, labels_true = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probs = softmax(logits, axis=1)
    labels_bin = label_binarize(labels_true, classes=list(range(num_labels)))

    try:
        roc_auc = roc_auc_score(labels_bin, probs, multi_class="ovr", average="macro")
    except ValueError:
        roc_auc = float("nan")
    try:
        pr_auc = average_precision_score(labels_bin, probs, average="macro")
    except ValueError:
        pr_auc = float("nan")

    cm = confusion_matrix(labels_true, predictions)
    all_confusion_matrices.append(cm)

    return {
        "accuracy": accuracy_score(labels_true, predictions),
        "balanced_accuracy": balanced_accuracy_score(labels_true, predictions),
        "precision": precision_score(labels_true, predictions, average="weighted", zero_division=0),
        "recall": recall_score(labels_true, predictions, average="weighted", zero_division=0),
        "f1": f1_score(labels_true, predictions, average="weighted", zero_division=0),
        "mcc": matthews_corrcoef(labels_true, predictions),
        "k_score": cohen_kappa_score(labels_true, predictions),
        "log_loss": log_loss(labels_true, probs, labels=list(range(num_labels))),
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
    }

###Δημιουργία data collator

In [ ]:
class DataCollatorCTCWithPadding:
    def __init__(self, feature_extractor, padding=True):
        self.feature_extractor = feature_extractor
        self.padding = padding

    def __call__(self, features):
        input_features = [{"input_values": f["input_values"]} for f in features]
        batch = self.feature_extractor.pad(input_features, padding=self.padding, return_tensors="pt")
        batch["labels"] = torch.tensor([f["labels"] for f in features], dtype=torch.long)
        return batch

data_collator = DataCollatorCTCWithPadding(feature_extractor=feature_extractor, padding=True)

###Ορισμός υπερπαραμέτρων και εκπαίδευση

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="./unispeech-sat-finetuned-IEMOCAP-4class",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    num_train_epochs=8,
    fp16=False,
    learning_rate=3e-5,
    warmup_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

In [ ]:
#Εκτύπωση των τελικών αποτελεσμάτων της εκπαίδευσης
eval_results = trainer.evaluate()
print(f"Αποτελέσματα αξιολόγησης: {eval_results}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

#Αποθήκευση των ετικετών για τους άξονες των matrices
labels = [
    model.config.id2label[i]
    for i in sorted(model.config.id2label.keys())
]

#Loop στην λίστα με τα confusion matrices
for epoch, cm in enumerate(all_confusion_matrices):
  #Δημιοργία γραφήματος
  plt.figure(figsize=(6,5))

  #Δηνιουργία Heatmap
  sns.heatmap(
      cm,
      annot=True,
      fmt="d",
      xticklabels=labels,
      yticklabels=labels
  )

  plt.title(f"Confusion Matrix - Epoch {epoch+1}")
  plt.xlabel("Predicted")
  plt.ylabel("True")
  plt.show()

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

In [ ]:
repo_id = "VasilisAsim/unispeech_sat-finetuned-IEMOCAP"

#Αποθήκευση του μοντέλου στο HuggingFace
model.push_to_hub(repo_id)

#Αποθήκευση του Feature Extractor
feature_extractor.push_to_hub(repo_id)

print(f"Το μοντέλο και ο επεξεργαστής αποθηκεύτηκαν στο Hugging Face Hub με ID: {repo_id}")